# EvoHash Stage 2 — NeuralHash optimizer

Same infrastructure as `pHash_optimize.ipynb`, adapted for NeuralHash:
- Loads seeds from `attack_configs.json` (WIP, not `_optimized`)
- CPU provider + process backend (CPU×30 >> GPU×8 on NeuralHash — verified)
- `CATEGORY_CHOICES` based on Spearman analysis from stage-1 search:
  `scale` fixed=global, `beta1/beta2/eps` fixed, `freq_dims` fixed=16,
  `max_freq` around base in [4,6,8,12,16], `n_samples`/`q` fixed=20
- `process_start_method='spawn'` on Linux (fork+CUDA = crash)

In [ ]:
from pathlib import Path
import importlib
import json
import os
import platform
import sys

# CPU-provider env vars must be set BEFORE importing onnxruntime / neuralhash.
# Workers inherit this env, so CPU provider + 1-thread-each avoids oversubscription.
os.environ['NEURALHASH_PROVIDER'] = 'cpu'
os.environ.setdefault('NEURALHASH_INTRA_THREADS', '1')
os.environ.setdefault('NEURALHASH_INTER_THREADS', '1')

os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')
os.environ.setdefault('NUMEXPR_NUM_THREADS', '1')

def find_project_root() -> Path:
    """Walk up from cwd to the repo folder (the one holding optimizer.py)."""
    for d in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (d / 'optimizer.py').is_file() and (d / 'stage2_optimize').is_dir():
            return d
    raise RuntimeError(f'Project root not found above {Path.cwd()}')

PROJECT_ROOT = find_project_root()
PACKAGE_PARENT = PROJECT_ROOT.parent
if str(PACKAGE_PARENT) not in sys.path:
    sys.path.insert(0, str(PACKAGE_PARENT))

import evohash.optimizer as opt
opt = importlib.reload(opt)

all_trials_dataframe = opt.all_trials_dataframe
comparison_dataframe = opt.comparison_dataframe
load_attack_config = opt.load_attack_config
print_tuned_configs = opt.print_tuned_configs
resolve_parallel_runtime = opt.resolve_parallel_runtime
run_hash_tuning_experiment = opt.run_hash_tuning_experiment
tuned_configs_dict = opt.tuned_configs_dict
tuning_plan_dataframe = opt.tuning_plan_dataframe

from evohash.dataset import Dataset
from evohash.optimizer import OptimizerSettings

print(f'Loaded optimizer from: {opt.__file__}')
print(f'Project root: {PROJECT_ROOT}')
print(f'Platform: {platform.system()} | NEURALHASH_PROVIDER={os.environ["NEURALHASH_PROVIDER"]}')

## Run target

In [ ]:
HASH_ID = 'neuralhash'

# 'all' or a list like ['atkscopes', 'simba'].
TUNE_ATTACKS_SETTING = 'all'
# TUNE_ATTACKS_SETTING = ['atkscopes', 'simba']

MAX_PAIRS = 30
BUDGET = 20_000
N_TRIALS = 200      # NeuralHash is slow; 200 trials is realistic per attack
SEED = 0
SPLIT = 'public'

# WIP configs (not yet 'optimized' — that file is for fully validated configs).
CONFIG_PATH = PROJECT_ROOT / 'results' / 'attack_configs.json'
ZIP_CANDIDATES = [
    PROJECT_ROOT / 'data' / 'dataset_v1_public.zip',
    PROJECT_ROOT.parent / 'dataset_v1_public.zip',  # legacy: next to the repo folder
]
ZIP_PATH = next((p for p in ZIP_CANDIDATES if p.exists()), ZIP_CANDIDATES[0])

OUT_DIR = PROJECT_ROOT / 'outputs' / f'parallel_around_config_optimizer_{HASH_ID}'
OUT_DIR.mkdir(parents=True, exist_ok=True)
STORAGE_PATH = OUT_DIR / f'{HASH_ID}_optuna.sqlite3'
STORAGE = f'sqlite:///{STORAGE_PATH.as_posix()}'

config = load_attack_config(CONFIG_PATH)
available_attacks = list(config['configs'][HASH_ID].keys())
if TUNE_ATTACKS_SETTING == 'all':
    TUNE_ATTACKS = available_attacks
else:
    TUNE_ATTACKS = list(TUNE_ATTACKS_SETTING)

missing = [a for a in TUNE_ATTACKS if a not in available_attacks]
if missing:
    raise ValueError(f'Unknown attacks for {HASH_ID}: {missing}. Available: {available_attacks}')

PAIR_COUNT = len(Dataset(str(ZIP_PATH), split=SPLIT, max_pairs=MAX_PAIRS))

print('HASH_ID:', HASH_ID)
print('TUNE_ATTACKS:', TUNE_ATTACKS)
print('ZIP_PATH:', ZIP_PATH, 'exists:', ZIP_PATH.exists())
print('PAIR_COUNT:', PAIR_COUNT)
print('BUDGET:', BUDGET, 'N_TRIALS:', N_TRIALS)
print('OUT_DIR:', OUT_DIR)

## Parallel settings

In [ ]:
PARALLEL_BACKEND = 'auto'   # auto -> process on all platforms for neuralhash
PARALLEL_WORKERS = None     # None -> optimizer resolves (Windows=8, Linux=30)
PARALLEL_WORKER_CAP = 30
PARALLEL_MIN_PAIRS = 2
PARALLEL_SHOW_PROGRESS = False

# NeuralHash: force spawn even on Linux — fork after CUDA init crashes workers.
# CPU provider is used (set above), but spawn is safer and still fast.
PROCESS_START_METHOD = 'spawn'

preview_settings = OptimizerSettings(
    hash_id=HASH_ID,
    attack_id=TUNE_ATTACKS[0],
    budget=BUDGET,
    seed=SEED,
    parallel_backend=PARALLEL_BACKEND,
    parallel_workers=PARALLEL_WORKERS,
    parallel_worker_cap=PARALLEL_WORKER_CAP,
    parallel_min_pairs=PARALLEL_MIN_PAIRS,
    parallel_show_progress=PARALLEL_SHOW_PROGRESS,
    parallel_process_start_method=PROCESS_START_METHOD,
)
runtime = resolve_parallel_runtime(preview_settings, PAIR_COUNT)
print('Resolved parallel runtime:', runtime)

## Category choices (NeuralHash-specific)

Based on stage-1 Spearman analysis. Key differences from pHash/PDQ:
- `scale=global` fixed (global>>mid, median 0.71 vs 0.34)
- `max_freq` around base but in **lower range** [4,6,8,12,16] (ρ=−0.38: lower = better on CNN)
- `beta1/beta2/eps` fixed (NaN correlation + free perturbation gave invalid beta2>1)
- `simba.freq_dims` fixed=16 (ρ=+0.008: no effect)
- `simba.epsilon` free numeric (ρ=+0.981: still climbing at 72)
- `n_samples`, `q` fixed=20 (only trade L2↔queries, not SR)
- `prokos.antithetic` fixed=True (True>False in categorical analysis)

In [ ]:
# 'around' specs — same mechanism as pHash/PDQ notebooks.
NS_FIXED_20    = [20]
Q_FIXED_20     = [20]

# max_freq: around base but clamped to the relevant CNN band [4,16].
# (Low 2/3 are pHash/PDQ-specific DCT values, not relevant for CNN.)
MAX_FREQ_AROUND_CNN = {'mode': 'around', 'radius': 2, 'step': 2,
                        'minimum': 4, 'maximum': 16}

CATEGORY_CHOICES = {
    'atkscopes': {
        'scale':    ['global'],           # fixed: global >> mid (Spearman)
        'max_freq': MAX_FREQ_AROUND_CNN,  # around base, CNN band [4,16]
        'beta1':    [0.9],                # fixed: NaN corr, perturb broke it
        'beta2':    [0.999],              # fixed: same
        'eps':      [1e-8],               # fixed
    },
    'simba': {
        'freq_dims': [16],  # fixed: ρ=+0.008, no effect; epsilon is the lever
        'stride':    [1],
    },
    'prokos': {
        'n_samples':       NS_FIXED_20,
        'antithetic':      [True],        # fixed: True always better in stage-1
        'grayscale_noise': [True],
    },
    'nes_attack_v0': {
        'n_samples':       NS_FIXED_20,
        'grayscale_noise': [True],
    },
    'nes': {
        'n_samples': NS_FIXED_20,
    },
    'zo_signsgd': {
        'q':                  Q_FIXED_20,
        'estimator':          ['forward', 'majority', 'central'],  # forward/majority > central
        'direction_dist':     ['gaussian'],
        'eval_updated_point': [True],
    },
}
NUMERIC_PARAMS = {}   # tune all supported continuous params
FIXED_PARAMS = {}

plan = tuning_plan_dataframe(
    config=config, hash_id=HASH_ID, attack_ids=TUNE_ATTACKS,
    category_choices=CATEGORY_CHOICES,
    numeric_params=NUMERIC_PARAMS, fixed_params=FIXED_PARAMS,
)
plan_path = OUT_DIR / 'tuning_plan.csv'
plan.to_csv(plan_path, index=False, encoding='utf-8')
print('Saved:', plan_path)
display(plan)

## Run optimization

In [ ]:
results = run_hash_tuning_experiment(
    config_path=CONFIG_PATH,
    zip_path=ZIP_PATH,
    hash_id=HASH_ID,
    attack_ids=TUNE_ATTACKS,
    split=SPLIT,
    max_pairs=MAX_PAIRS,
    budget=BUDGET,
    n_trials=N_TRIALS,
    seed=SEED,
    asr_floor=0.05,
    failure_gap_penalty=0.0,
    category_choices=CATEGORY_CHOICES,
    numeric_params=NUMERIC_PARAMS,
    fixed_params=FIXED_PARAMS,
    study_prefix=f'nh_around_budget{BUDGET}_pairs{MAX_PAIRS}',
    storage=STORAGE,
    rerun_best=False,
    verbose=True,
    parallel_backend=PARALLEL_BACKEND,
    parallel_workers=PARALLEL_WORKERS,
    parallel_worker_cap=PARALLEL_WORKER_CAP,
    parallel_min_pairs=PARALLEL_MIN_PAIRS,
    parallel_show_progress=PARALLEL_SHOW_PROGRESS,
    parallel_process_start_method=PROCESS_START_METHOD,
)

## Results and exports

In [ ]:
comparison_df = comparison_dataframe(results)
trials_df = all_trials_dataframe(results)
configs = tuned_configs_dict(results)

comparison_path = OUT_DIR / 'comparison.csv'
trials_path = OUT_DIR / 'trials.csv'
configs_path = OUT_DIR / 'tuned_configs.json'

comparison_df.to_csv(comparison_path, index=False, encoding='utf-8')
trials_df.to_csv(trials_path, index=False, encoding='utf-8')
configs_path.write_text(json.dumps(configs, indent=2, sort_keys=True), encoding='utf-8')

print('Saved:', comparison_path)
print('Saved:', trials_path)
print('Saved:', configs_path)
display(comparison_df)

In [ ]:
print_tuned_configs(results)

best_configs = tuned_configs_dict(results)
best_configs_path = OUT_DIR / 'best_configs.json'
best_configs_path.write_text(json.dumps(best_configs, indent=2, sort_keys=True), encoding='utf-8')
print('\nBest configs JSON')
print(json.dumps(best_configs, indent=2, sort_keys=True))

# Promote to attack_configs.json if improved (NOT _optimized yet — needs validation).
PROMOTE = False  # True rewrites results/attack_configs.json
if PROMOTE:
    _cfg_all = load_attack_config(str(CONFIG_PATH))
    for atk_id, new_params in best_configs[HASH_ID].items():
        _cfg_all['configs'][HASH_ID][atk_id] = new_params
    CONFIG_PATH.write_text(json.dumps(_cfg_all, indent=2, sort_keys=True), encoding='utf-8')
    print(f'Promoted {len(best_configs[HASH_ID])} configs -> {CONFIG_PATH.name}')